In [0]:
# Databricks notebook source
# COMMAND ----------
# DBTITLE 1,Load Audit Utility & Run Context
%run ./00_audit_utilsnb.ipynb

# COMMAND ----------
import traceback
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, to_date

# Retrieve run_id from shprod.audit_runlog
run_id = int(spark.sql("SELECT COALESCE(MAX(run_id), 1) FROM shprod.audit_runlog").collect()[0][0])

print(f"run_id = {run_id}")

Warning you are using the ipython `%run` line magic. To use the databricks `%run` cell magic make sure that the magic is at the very start of the cell.

run_id = 3


In [0]:
logger = PipelineLogger(spark=spark, pipeline_name="NYC_PAYROLL_BATCH_SCD_PIPELINE", run_id=run_id)
logger.open_runlog()

# Set task value for downstream tasks
dbutils.jobs.taskValues.set(key="pipeline_run_id", value=run_id)


In [0]:
step_start = datetime.now()
target_table_name = "shprod.payroll_employee_nyc"
source_table_name = "shprod.stg_payroll_employee_nyc"

In [0]:
#SCD Steps Begin
try:
    stg_df = spark.table(source_table_name)
    target_table = DeltaTable.forName(spark, target_table_name)
    
    # Active records check using the sentinel date
    target_active_df = target_table.toDF().filter("end_dt = '9999-12-31'")

    # Leg 1: Changed active records (to be closed out)
    changed_records_df = (
        stg_df.alias("s")
        .join(target_active_df.alias("t"), col("s.pid") == col("t.pid"), "inner")
        .filter(
            (col("s.agency_name").eqNullSafe(col("t.agency_name"))) |
            (col("s.title_description").eqNullSafe(col("t.title_description"))) |
            (col("s.work_location_borough").eqNullSafe(col("t.work_location_borough"))) |
            (col("s.leave_status_as_of_june_30").eqNullSafe(col("t.leave_status_as_of_june_30"))) |
            (col("s.base_salary").eqNullSafe(col("t.base_salary"))) |
            (col("s.pay_basis").eqNullSafe(col("t.pay_basis"))) |
            (col("s.regular_hours").eqNullSafe(col("t.regular_hours"))) |
            (col("s.regular_gross_paid").eqNullSafe(col("t.regular_gross_paid"))) |
            (col("s.ot_hours").eqNullSafe(col("t.ot_hours"))) |
            (col("s.total_ot_paid").eqNullSafe(col("t.total_ot_paid"))) |
            (col("s.total_other_pay").eqNullSafe(col("t.total_other_pay")))
        )
        .select("s.*")
        .withColumn("merge_key", col("s.pid"))
        #.dropDuplicates(["pid"])
    )

    # Leg 2: Brand-new records + new versions of changed records
    insert_records_df = stg_df.withColumn("merge_key", lit(None).cast("string"))

    # Union both streams
    staged_updates_df = changed_records_df.unionByName(insert_records_df)

    # Atomic Delta MERGE
    (
        target_table.alias("target")
        .merge(
            staged_updates_df.alias("source"),
            "target.pid = source.merge_key AND target.end_dt = CAST('9999-12-31' AS DATE)"
        )
        # Leg 1 Match: Retire previous active row
        .whenMatchedUpdate(
            set={
                "end_dt": to_date(col("source.active_dt")),
                "lst_uptd_run_id": col("source.run_id")
            }
        )
        # Leg 2 No-Match: Insert new active row with sentinel end date
        .whenNotMatchedInsert(
            values={
                "pid": col("source.pid"),
                "fiscal_year": col("source.fiscal_year"),
                "agency_name": col("source.agency_name"),
                "first_name": col("source.first_name"),
                "last_name": col("source.last_name"),
                "mid_name": col("source.mid_name"),
                "agency_start_date": col("source.agency_start_date"),
                "work_location_borough": col("source.work_location_borough"),
                "title_description": col("source.title_description"),
                "leave_status_as_of_june_30": col("source.leave_status_as_of_june_30"),
                "base_salary": col("source.base_salary"),
                "pay_basis": col("source.pay_basis"),
                "regular_hours": col("source.regular_hours"),
                "regular_gross_paid": col("source.regular_gross_paid"),
                "ot_hours": col("source.ot_hours"),
                "total_ot_paid": col("source.total_ot_paid"),
                "total_other_pay": col("source.total_other_pay"),
                "active_dt": col("source.active_dt"),
                "end_dt": to_date(lit("9999-12-31")),
                "run_id": col("source.run_id"),
                "lst_uptd_run_id": col("source.run_id")
            }
        )
        .execute()
    )
# Metrics collection & audit logging
    last_op_metrics = target_table.history(1).select("operationMetrics").collect()[0][0]
    num_inserted = int(last_op_metrics.get("numTargetRowsInserted", 0))
    num_updated = int(last_op_metrics.get("numTargetRowsUpdated", 0))

    logger.log_step(
        step_name="02_SCD2_MERGE",
        source_table=source_table_name,
        target_table=target_table_name,
        error_table="payroll_quarantine",
        status="SUCCESS",
        rows_read=stg_df.count(),
        rows_inserted=num_inserted,
        rows_updated=num_updated,
        start_time=step_start,
        end_time=datetime.now()
    )
    logger.close_runlog()


except Exception as e:
    err_msg = str(traceback.format_exc()).replace("'", "\"")[:1000]
    logger.log_step(
        step_name="02_SCD2_MERGE",
        source_table=source_table_name,
        target_table=target_table_name,
        error_table="payroll_quarantine",
        status="FAILED",
        start_time=step_start,
        end_time=datetime.now(),
        error_msg=err_msg
    )
    logger.close_runlog(status="FAILED")
    raise e